# London Property Intelligence

## Portfolio Project

### Project Overview

This project analyzes residential property transactions across London's 33 administrative areas using HM Land Registry data.

The analysis explores market activity, property prices, and market opportunities through exploratory data analysis, feature engineering, and business-driven scoring methods.

---

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
PROJECT_ROOT = Path.cwd().parent
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "pp-2025.csv"
DATA_PATH

In [ ]:
df = pd.read_csv(DATA_PATH)

df

In [ ]:
df.columns

In [ ]:
column_names = [
    "Transaction unique identifier",
    "Price",
    "Date of Transfer",
    "Postcode",
    "Property Type",
    "Old/New",
    "Duration",
    "PAON",
    "SAON",
    "Street",
    "Locality",
    "Town/City",
    "District",
    "County",
    "PPD Category Type",
    "Record Status"
]

df = pd.read_csv(
    DATA_PATH,
    header=None,
    names=column_names
)

df.head()

In [ ]:
df["County"].value_counts().head(30)

## Data Understanding

### Question

Can the **District** column be used to identify London boroughs?

### Why does this matter?

The project focuses exclusively on London property investment.

Before filtering the dataset, we need to verify whether the **District** column contains the geographical information required to identify London boroughs.

### Expected outcome

This analysis will determine whether the **District** column can be used as the primary geographic identifier for constructing the London-only dataset.

## Creating the London Dataset

### Objective

The original HM Land Registry dataset contains transactions from across England and Wales.

Since the scope of this project is limited to London property investment, the first preprocessing step is to isolate London transactions.

### Approach

For Version 1 of the project, London transactions are identified using the `County` field.

Records where:

County = "GREATER LONDON"

will be retained for further analysis.

### Output

The filtered dataset will be saved as:

data/processed/london_transactions.csv

In [ ]:
london_df = df[df["County"] == "GREATER LONDON"].copy()

print(f"Number of London transactions: {len(london_df):,}")

london_df.head()

In [ ]:
london_df["County"].value_counts()

### Validation

The filtering step was successfully validated.

All records in the filtered dataset belong to:

County = GREATER LONDON

This confirms that the London dataset has been correctly isolated from the original England and Wales transaction data.

In [ ]:
london_df["District"].value_counts().head(20)

### Conclusion

The dataset already provides London borough names in the `District` column.

Therefore:

- `County` will be used to filter London transactions.
- `District` will be used for borough-level analysis.

In [ ]:
OUTPUT_PATH = PROJECT_ROOT / "data" / "processed" / "london_transactions.csv"

OUTPUT_PATH

In [ ]:
london_df.to_csv(OUTPUT_PATH, index=False)

print("London dataset saved successfully.")

In [ ]:
saved_df = pd.read_csv(OUTPUT_PATH)

saved_df.head()

In [ ]:
saved_df.shape

In [ ]:
df["District"].value_counts().head(20)

# Exploratory Data Analysis (EDA)

## Business Question 1

Which London boroughs have the highest property transaction activity?

### Why does this matter?

Understanding transaction activity helps identify the most active property markets across London. Highly active boroughs may indicate greater market liquidity and provide a useful starting point for investment analysis.

### Expected outcome

Identify the boroughs with the highest number of recorded property transactions.

In [ ]:
borough_activity = (
    london_df["District"]
    .value_counts()
    .reset_index()
)

borough_activity.columns = ["Borough", "Transactions"]

borough_activity.head(10)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
top10 = borough_activity.head(10)

In [ ]:
plt.figure(figsize=(10, 6))
plt.barh(
    top10["Borough"],
    top10["Transactions"]
)
plt.xlabel("Number of Transactions")
plt.ylabel("London Borough")
plt.title("Top 10 London Boroughs by Property Transactions")
plt.tight_layout()

plt.show()

## Insight

Wandsworth recorded the highest number of property transactions in the dataset, followed by Croydon and Bromley. This suggests that these boroughs were among the most active property markets during the period covered by the dataset.

However, transaction volume alone should not be interpreted as an indicator of investment attractiveness. A higher number of transactions reflects market activity rather than profitability or future growth potential. Further analysis of property prices, price trends, and other market indicators is required before drawing investment conclusions.

## Business Question 2

### Question

Which London boroughs have the highest average property prices?

### Why does this matter?

Transaction activity shows how active the market is, but it does not indicate property values.

Average property prices provide a clearer picture of the relative cost of buying property across London boroughs and form an important foundation for investment analysis.

### Expected outcome

Calculate and compare the average property price for each London borough to identify the most expensive areas in the dataset.


In [ ]:
average_price = (
    london_df
    .groupby("District")["Price"]
    .mean()
    .reset_index()
)

average_price.columns = ["Borough", "Average Price"]

average_price.head()


In [ ]:
average_price = average_price.sort_values(
    by="Average Price",
    ascending=False
)

average_price.head(10)


In [ ]:
top10_price = average_price.head(10)

top10_price

In [ ]:
plt.figure(figsize=(10, 6))

plt.barh(
    top10_price["Borough"],
    top10_price["Average Price"]
)

plt.title("Top 10 London Boroughs by Average Property Price")
plt.xlabel("Average Property Price (£)")
plt.ylabel("London Borough")

plt.tight_layout()

plt.show()

### Insight

The highest average property prices are concentrated in Central London boroughs, with the City of London, the City of Westminster, and Kensington and Chelsea ranking at the top.

A noticeable price gap exists between the top three boroughs and the remaining boroughs, suggesting that London's premium property market is highly concentrated in a relatively small number of central locations.

Average property price alone does not fully describe market attractiveness. It should be interpreted alongside transaction activity to provide a more balanced understanding of each borough's property market.

## Business Question 3

Which London boroughs offer the best balance between property value and market activity?

### Why does this matter?

Investors need to consider both property prices and market activity when evaluating potential locations. High property prices may indicate premium markets, while strong transaction activity may suggest greater market liquidity and demand. Considering both factors provides a more balanced view of investment opportunities across London boroughs.

### Expected outcome

Identify boroughs that demonstrate a strong combination of average property prices and transaction activity.

In [ ]:
market_summary = pd.merge(
    borough_activity,
    average_price,
    on="Borough"
)

market_summary.head()

In [ ]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

market_summary["Transactions Scaled"] = scaler.fit_transform(
    market_summary[["Transactions"]]
)

market_summary.head()

In [ ]:
market_summary["Price Scaled"] = scaler.fit_transform(
    market_summary[["Average Price"]]
)

market_summary.head()

In [ ]:
market_summary[["Borough", "Average Price", "Price Scaled"]].head(10)

In [ ]:
market_summary.sort_values(
    by="Average Price",
    ascending=False
).head(10)

In [ ]:
import numpy as np

market_summary["Log Average Price"] = np.log(
    market_summary["Average Price"]
)

market_summary.head()

In [ ]:
market_summary["Log Price Scaled"] = scaler.fit_transform(
    market_summary[["Log Average Price"]]
)

market_summary.head()

## Feature Engineering: Handling Property Price Skewness

### Problemُ

Average property prices showed a strong right-skewed distribution, mainly due to a few premium central London boroughs with extremely high prices.

Examples:
- City of London
- City of Westminster
- Kensington and Chelsea

Using raw prices directly could make these areas dominate the scoring process.

### Approach

I applied a log transformation to the Average Price feature before scaling.

Transformation:

Average Price → Log Average Price → Log Price Scaled

### Reason

The log transformation reduces the impact of extreme values while keeping the relative ranking of boroughs.

For the Market Opportunity Score, Log Price Scaled will be used instead of the original Price Scaled feature.

In [ ]:
market_summary.sort_values(
    by="Log Price Scaled",
    ascending=False
).head(10)

## Checking Log Price Scaling

After applying log transformation and scaling, the highest-value boroughs were reviewed to confirm that premium areas still maintain higher scores while reducing the impact of extreme prices.

In [ ]:
market_summary["Market Opportunity Score"] = (
    0.5 * market_summary["Transactions Scaled"]
    +
    0.5 * market_summary["Log Price Scaled"]
)

market_summary.head()

## Creating Market Opportunity Score

To identify boroughs with better investment opportunities, two factors were combined:

- Market activity: Transactions Scaled
- Property value: Log Price Scaled

Both features were given equal importance (50% each) because the objective is to find a balance between market activity and property value rather than focusing only on premium locations.

### Formula

Market Opportunity Score =
0.5 × Transactions Scaled +
0.5 × Log Price Scaled

In [ ]:
market_summary_sorted = market_summary.sort_values(
    by="Market Opportunity Score",
    ascending=False
)

market_summary_sorted.head(10)

## Ranking Boroughs by Market Opportunity Score

Boroughs were ranked based on the Market Opportunity Score, which combines property value and market activity.

The ranking should be interpreted as an initial market opportunity indicator based on available transaction data, rather than a complete investment recommendation.

Additional factors such as rental yield, price growth, and economic indicators would be required for a more comprehensive investment analysis.

In [ ]:
top_boroughs = market_summary_sorted.head(10)

top_boroughs

In [ ]:
top_boroughs_plot = top_boroughs[
    ["Borough", "Market Opportunity Score"]
]

top_boroughs_plot

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

plt.barh(
    top_boroughs_plot["Borough"],
    top_boroughs_plot["Market Opportunity Score"]
)

plt.xlabel("Market Opportunity Score")
plt.ylabel("Borough")
plt.title("Top 10 London Boroughs by Market Opportunity Score")

plt.gca().invert_yaxis()

plt.show()

## Visualising Top Market Opportunities

The top 10 boroughs were visualised based on the Market Opportunity Score to highlight areas with a stronger balance between property value and market activity.

In [ ]:
id="k2m8p"
top_boroughs_analysis = market_summary_sorted.head(10)[
    [
        "Borough",
        "Transactions",
        "Average Price",
        "Market Opportunity Score"
    ]
]

top_boroughs_analysis

## Sensitivity Analysis: Testing Different Feature Weights

The initial Market Opportunity Score used equal weights for market activity and property value (50/50).

To understand how sensitive the ranking is to different priorities, alternative weighting scenarios are tested.

For example, an activity-focused scenario gives more importance to transaction volume:

70% Market Activity  
30% Property Value

This helps evaluate whether the identified opportunities remain consistent under different investment preferences.

In [ ]:
market_summary["Activity Focused Score"] = (
    0.7 * market_summary["Transactions Scaled"]
    +
    0.3 * market_summary["Log Price Scaled"]
)

market_summary.sort_values(
    by="Activity Focused Score",
    ascending=False
).head(10)

## Sensitivity Analysis Result

Changing the feature weights affected the ranking of boroughs.

When market activity received higher importance (70/30 scenario), highly active markets such as Wandsworth, Bromley, and Croydon moved higher in the ranking.

This demonstrates that the Market Opportunity Score reflects different investor priorities rather than representing a single universal ranking.

In [ ]:
score_comparison = market_summary[
    [
        "Borough",
        "Market Opportunity Score",
        "Activity Focused Score"
    ]
]


In [ ]:
market_summary["Robust Opportunity Score"] = (
    market_summary["Market Opportunity Score"]
    +
    market_summary["Activity Focused Score"]
) / 2

market_summary.sort_values(
    by="Robust Opportunity Score",
    ascending=False
).head(10)

## Robust Opportunity Score

To identify boroughs that remain strong under different investor preferences, a Robust Opportunity Score was created.

This score represents the average performance across two scenarios:

- Balanced scenario (50% market activity, 50% property value)
- Activity-focused scenario (70% market activity, 30% property value)

Boroughs with higher Robust Opportunity Scores demonstrate more consistent performance across different investment priorities.

In [ ]:
top_robust_boroughs = market_summary.sort_values(
    by="Robust Opportunity Score",
    ascending=False
).head(10)

top_robust_boroughs

In [ ]:
top_robust_plot = top_robust_boroughs[
    ["Borough", "Robust Opportunity Score"]
]

top_robust_plot

## Final Ranking: Robust Market Opportunities

The final ranking is based on the Robust Opportunity Score, which combines multiple investor perspectives.

This metric identifies boroughs that maintain strong performance across different weighting scenarios, providing a more stable view of potential market opportunities.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))

bars = plt.barh(
    top_robust_plot["Borough"],
    top_robust_plot["Robust Opportunity Score"]
)

plt.xlabel("Robust Opportunity Score")
plt.ylabel("Borough")
plt.title("Top 10 London Boroughs by Robust Opportunity Score")

plt.gca().invert_yaxis()

# Add score labels
for bar in bars:
    score = bar.get_width()
    plt.text(
        score + 0.01,
        bar.get_y() + bar.get_height()/2,
        f"{score:.3f}",
        va="center"
    )

plt.show()

## Business Question 3 — Final Insight

The analysis aimed to identify London boroughs that provide a strong balance between property value and market activity.

A Market Opportunity Score was developed by combining transaction volume and property value. To improve the robustness of the ranking, sensitivity analysis was performed by testing different feature weight combinations.

The final Robust Opportunity Score identified boroughs that maintained strong performance across different investor preferences.

Wandsworth achieved the highest Robust Opportunity Score, reflecting a strong combination of market activity and property value across the evaluated scenarios. City of Westminster also showed strong performance, mainly driven by its premium property values.

Other boroughs such as Bromley, Barnet, and Croydon demonstrated strong market activity combined with more moderate property prices, highlighting different types of potential market opportunities.

These results should be interpreted as an initial market opportunity indicator based on transaction activity and property prices. Additional factors such as rental yield, price growth, economic indicators, and local characteristics would be required for a comprehensive investment analysis.